# DWthon Memory - Dzień 2: `Nie dokładaj tokenów. Zaprojektuj pamięć`

Witaj z powrotem. Dzień 1 był konceptualny - narysowaliśmy architekturę na papierze i posortowałeś fiszki Marka do 7 szuflad. Zobaczysz, że ten sam model raz jest amnezjakiem, a raz "starym znajomym". Dziś schodzimy z konceptu do kodu i **domykamy całą pętlę pamięci**: agent dostaje wiadomość, aktualizuje stan, zasila nim prompt, odpowiada, a na koniec sesji kompresuje rozmowę i zapisuje wnioski na przyszłość.

> Na koniec dnia będziesz mieć działający model stanu, który sam się aktualizuje i świadomie zasila prompt. Nie slajd o pamięci - działającą pamięć.

## Czego nauczysz się dzisiaj?

1. Poznasz **4 operacje na pamięci** - `Write`, `Select`, `Compress`, `Isolate` - i zobaczysz każdą w działającym kodzie
2. Przełożysz **warstwy pamięci na struktury Pydantic** i zbudujesz `AgentState` - jedno źródło prawdy o agencie
3. Dodasz warstwę **LearningProgress** - dzięki niej Tutor pamięta nie tylko fakty, ale i **postęp** ucznia
4. Zobaczysz `build_prompt(state)` - czyli **świadome zasilanie promptu pamięcią** + mechanikę **KK ("Końskie Klapki")**
5. Uruchomisz `run_turn(...)` i poczujesz, jak pamięć **realnie** zmienia odpowiedź modelu
6. Skompresujesz chaotyczną rozmowę w czystą strukturę (spłacamy obietnicę z Dnia 1: "naiwne streszczanie zawodzi - a jak zrobić to dobrze?")
7. **Zamkniesz pętlę** - epizod wróci do stanu, zaktualizuje postęp i zasili następny prompt

## Aha moment w 20 sekund

Zanim wjedzie kod, zobacz różnicę po ludzku:

```text
❌ CHATBOT-AMNEZJAK              ✅ TUTOR Z PAMIĘCIĄ
──────────────────              ────────────────────
Uczeń: Cześć!                   Uczeń: Cześć!
Bot:   Dzień dobry, w czym      Tutor: Cześć Marek! Jak Luna?
       mogę pomóc?                     Ostatnio ćwiczyliśmy
(nie wie kim jesteś,                   otwarcie prezentacji –
 zaczyna od zera za                    gotowy na dalszy ciąg
 każdym razem)                         przed New York?

       ▲                                     ▲
brak build_prompt(state)          state wstrzyknięty do promptu
```

To właśnie zbudujemy dzisiaj: **nie więcej tokenów, tylko lepszą architekturę pamięci**.


## Zanim zaczniesz - ścieżka minimum (przeczytaj, to obniża stres)

Dziś jest sporo kodu. Ale spokojnie - **nie musisz znać całego Pythona ani wszystkiego przepisywać**. Trzymaj się tej zasady:

- ▶️ **Uruchamiaj wszystkie komórki po kolei** (Shift + Enter). Większość kodu jest gotowa - jest po to, żebyś **zobaczył mechanikę**, a nie żebyś ją wkuwał.
- **Samodzielnie uzupełniasz tylko dwa miejsca:** Zadanie 2.1 i Zadanie 2.2. Reszta jedzie z automatu.
- **Jeśli masz patrzeć na cokolwiek uważnie - patrz na 4 funkcje:** `build_prompt`, `run_turn`, `compress_episode`, `apply_episode_to_state`. To one niosą cały dzień. Będę je wyraźnie zaznaczał.

Tyle. Jak zgubisz się w składni - to normalne. Idź dalej, efekt i tak zobaczysz.


## Ściąga do składni

Jeśli Python nie jest Twoim codziennym narzędziem, wystarczy, że rozpoznasz 6 rzeczy - resztę zrozumiesz z kontekstu:

| Zapis | Po ludzku |
|---|---|
| `str` | tekst |
| `List[str]` | lista tekstów |
| `Optional[str]` | tekst **albo** nic (`None`) |
| `BaseModel` | kontener na dane (taki "formularz") |
| `Field(...)` | opis pola - podpowiedź dla nas i dla modelu |
| `Enum` | zamknięta lista dozwolonych opcji - np. faza może być **tylko** jedną z 3 wartości |

To wszystko. Możemy jechać.

## 7 warstw pamięci

Zanim zejdziemy do kodu, zróbmy **krótkie odświeżenie z Dnia 1**. To jest ta sama mapa, tylko teraz zaczniemy ją przekładać na działające struktury.


<div style="text-align: center;">
  <img src="../images/memory_blue_print.jpg" width="500" alt="Memory Blueprint">
</div>


| Warstwa | Najkrócej: za co odpowiada? | Dziś używamy? |
|---|---|---|
| 🏛️ **Core** | kim jest agent i czego mu nie wolno łamać | ✅ tak |
| ⚡ **Working** | co padło przed chwilą w tej sesji | ✅ tak |
| 🎯 **Workflow** | na jakim etapie procesu jesteśmy | ✅ tak |
| 📚 **Episodic** | co wydarzyło się w poprzednich sesjach | ✅ tak |
| 🧠 **Semantic** | trwała wiedza o użytkowniku *(dziś: profil + postęp)* | ✅ tak |
| 🛠️ **Procedural** | jak wykonać zadanie / jakie ma procedury | ❌ nie dziś |
| 🔧 **Resource** | czy narzędzia i zasoby działają | ❌ nie dziś |

☝️ **Ważne:** dziś świadomie implementujemy **5 z 7 warstw**. To w zupełności wystarczy, żeby dowieźć obietnicę DWthonu: agent ma pamiętać **użytkownika, kontekst i postęp**. Warstwy `Procedural` i `Resource` dołożymy dopiero w pełnej ścieżce.

**Dodatkowa uwaga:** w tym notebooku warstwę `Semantic` rozbijamy praktycznie na dwie "szuflady": **`StudentProfile`** (kim jesteś) i **`LearningProgress`** (jak Ci idzie). To celowe uproszczenie pod use case Tutora.




```text
   7 WARSTW (koncept z Dnia 1)        AGENT STATE (dzisiejszy kod)
   ───────────────────────────       ────────────────────────────
   🏛️  Core          ✅  ───────────►  state.core
   🧠  Semantic      ✅  ─────┬────►  state.profile    (kim jesteś)
                               └────►  state.progress   (jak Ci idzie)
   🎯  Workflow      ✅  ───────────►  state.workflow
   ⚡  Working        ✅  ───────────►  state.working
   📚  Episodic      ✅  ───────────►  state.episodic
   🛠️  Procedural    ❌  ───────────►  (pełna ścieżka: Agentic AI)
   🔧  Resource      ❌  ───────────►  (pełna ścieżka: Agentic AI)

   7 warstw w teorii  →  5 wdrażamy  →  Semantic rozbita na 2  =  6 pól w kodzie
```

To jest cały trik liczbowy. Nie ma tu sprzeczności:
- **7** warstw to pełna mapa konceptualna,
- **5** z nich wdrażamy dzisiaj,
- `Semantic` rozbijamy na **2 obiekty robocze**,
- dlatego `AgentState` kończy z **6 polami**.

## Od rzeczowników do czasowników

Wczoraj mówiliśmy o **rzeczownikach**: warstwy, szuflady, fiszki. To "co" pamiętamy. Ale sama struktura to martwe muzeum. Żeby pamięć ożyła, potrzebujemy **czasowników**. Cały dzień 2 kręci się wokół czterech operacji:

| Operacja | Po ludzku | Gdzie ją dziś zobaczysz |
|---|---|---|
| 🔴 **Write** | zapisz / zaktualizuj | `run_turn` + `apply_episode_to_state` |
| 🔵 **Select** | wybierz / pobierz | `build_prompt` + KK |
| 🟢 **Compress** | skompresuj | `compress_episode` |
| 🟠 **Isolate** | izoluj | `AgentState` - każda warstwa w swojej przegródce |

## Dlaczego kod, a nie od razu baza danych?

Zaraz padnie pytanie: *"Gdzie baza? Redis? Postgres? Baza wektorowa?"*. Odpowiedź: **na start żadna**. Pamiętasz wojnę o storage z Dnia 1? To wtórne. Najpierw struktura - co, kiedy i jak trzymasz. Baza to tylko mechanizm trwałości, który dokładasz później. Użyjemy **Pydantic**: struktura "na dłoni", walidacja za darmo, jedno miejsce, gdzie mieszka wiedza agenta.

## Wczytujemy biblioteki

In [ ]:
from enum import Enum
from typing import List, Optional, Dict
from pydantic import BaseModel, Field
from jinja2 import Template
from openai import OpenAI
from dotenv import load_dotenv
import instructor

## Inicjalizacja

In [ ]:
load_dotenv()

# Klient ze structured output - użyjemy go do tur rozmowy i do kompresji
client = instructor.patch(
    OpenAI(base_url="https://llm-api.dataworkshop.eu"),
    mode=instructor.Mode.MD_JSON
)

print("✅ Gotowe. Budujemy pamięć.")

## 5 warstw jako struktury (przejrzyj i uruchom)

Poniżej wszystkie warstwy jako klasy Pydantic, podzielone na **dwie krótkie porcje**. **Nie musisz analizować każdego pola** - przeleć wzrokiem, uruchom obie komórki i jedziemy dalej. Każda warstwa to osobna "szuflada" (nasza operacja **Isolate**).

Zapamiętaj tylko jedno koncepcyjnie: Semantic dzielimy na **`StudentProfile`** (trwałe fakty: kim jesteś) i **`LearningProgress`** (stan nauki: jak Ci idzie). Bez tej drugiej Tutor pamięta *kim jesteś*, ale nie pamięta *postępu*.

## Część 1/2: tożsamość + wiedza o uczniu
- 🏛️ `CORE`: tożsamość i nieprzekraczalne zasady
- 🧠 `SEMANTIC` (część 1): trwałe fakty o uczniu
- 🧠 `SEMANTIC` (część 2): postęp nauki

In [ ]:
# --- 🏛️ CORE: tożsamość i nieprzekraczalne zasady ---
class CoreMemory(BaseModel):
    name: str = "Tutor"
    creator_name: str = "DataWorkshop"
    role: str = "Language Communication Coach"
    prime_directives: List[str] = [
        "NO_HOMEWORK_SOLVING: nigdy nie podawaj gotowców - naprowadzaj",
        "COMMUNICATION_FIRST: priorytetem jest płynność, nie idealna gramatyka",
    ]

# --- 🧠 SEMANTIC (część 1): trwałe fakty o uczniu ---
class StudentProfile(BaseModel):
    user_id: str                                  # kotwica - klucz do bazy
    name: str
    level: str = Field(description="Poziom, np. B1")
    goal: str = Field(description="Po co się uczy")
    preferences: List[str] = Field(default_factory=list)
    context_notes: Optional[str] = None           # wydestylowane fakty, nie surowy log

# --- 🧠 SEMANTIC (część 2): postęp nauki ---
class LearningProgress(BaseModel):
    recent_mistakes: List[str] = Field(default_factory=list)   # błędy do przećwiczenia
    homework_status: str = "none"                              # none | assigned | done
    next_focus: Optional[str] = None                           # co ćwiczyć dalej

print("✅ Core + Semantic (profil + postęp) gotowe.")

## Część 2/2: proces, bieżąca rozmowa, epizody i spinacz wszystkiego
- 🎯 `WORKFLOW`: maszyna stanów (etap sesji)
- ⚡ `WORKING`: bieżąca rozmowa (sliding window)
- 📚 `EPISODIC`: skompresowane epizody z poprzednich sesji
- `AGENT STATE`: jedno źródło prawdy (spina wszystkie warstwy)

In [ ]:
# --- 🎯 WORKFLOW: maszyna stanów (etap sesji) ---
class SessionPhase(str, Enum):
    HOMEWORK_CHECK = "HOMEWORK_CHECK"
    PRACTICE = "PRACTICE"
    WRAP_UP = "WRAP_UP"

class WorkflowMemory(BaseModel):
    phase: SessionPhase = SessionPhase.HOMEWORK_CHECK
    topic: str

# --- ⚡ WORKING: bieżąca rozmowa (sliding window) ---
class WorkingMemory(BaseModel):
    history: List[Dict[str, str]] = Field(default_factory=list)

    def add(self, role: str, content: str):
        self.history.append({"role": role, "content": content})

    def get_context_window(self, limit: int = 6) -> List[Dict[str, str]]:
        return self.history[-limit:]   # bierzemy tylko ostatnie N wiadomości

# --- 📚 EPISODIC: skompresowane epizody z poprzednich sesji ---
class MistakeLog(BaseModel):
    user_phrase: str
    correction: str
    rule: str

class Episode(BaseModel):
    summary: str = Field(description="Jednozdaniowe podsumowanie sesji")
    personal_facts: List[str] = Field(default_factory=list)
    mistakes: List[MistakeLog] = Field(default_factory=list)
    next_step: str = Field(default="", description="Co ćwiczyć dalej")

class EpisodicMemory(BaseModel):
    episodes: List[Episode] = Field(default_factory=list)
    max_episodes: int = 10

    def add(self, episode: Episode):
        self.episodes.append(episode)
        self.episodes = self.episodes[-self.max_episodes:]   # wygaszanie: trzymamy ostatnie N

    # na dziś priorytetyzacja = recency: do promptu bierzemy tylko 2 ostatnie epizody
    def recent(self, limit: int = 2) -> List[Episode]:
        return self.episodes[-limit:]

# --- 🧩 AGENT STATE: jedno źródło prawdy (spina wszystkie warstwy) ---
class AgentState(BaseModel):
    core: CoreMemory
    profile: StudentProfile
    progress: LearningProgress
    workflow: WorkflowMemory
    working: WorkingMemory
    episodic: EpisodicMemory

print("✅ Workflow + Working + Episodic + AgentState gotowe.")

## Które warstwy pracują?

To jest ważne, żeby nie było wrażenia, że "narysowaliśmy 7 pudełek, a tak naprawdę działa tylko jedno". Nie. **Dziś kilka warstw pracuje naraz, tylko nie wszystkie zmieniają się w tym samym momencie.**

| Warstwa | Co robi dzisiaj? | Czy się dziś zmienia? |
|---|---|---|
| 🏛️ `CoreMemory` | ustawia tożsamość i zasady tutora w promptcie | ❌ nie - jest read-only |
| 🧠 `StudentProfile` | wnosi trwałe fakty o Marku (poziom, cel, preferencje) | ❌ nie - dziś tylko czytamy |
| 🧠 `LearningProgress` | wnosi pamięć o błędach i następnym kroku | ✅ tak - po zapisaniu epizodu |
| 🎯 `WorkflowMemory` | steruje fazą rozmowy i techniką KK | ✅ czasem - gdy zmieniasz `phase` |
| ⚡ `WorkingMemory` | zbiera bieżącą rozmowę tura po turze | ✅ tak - przy każdej turze |
| 📚 `EpisodicMemory` | przechowuje skompresowane epizody z sesji | ✅ tak - po `compress_episode` i `add()` |

Krótko: **po jednej turze od razu rusza `Working`**, `Workflow` steruje tym, co model widzi, a **po zamknięciu sesji ruszają `Episodic` i `LearningProgress`**. `Core` i `StudentProfile` przez cały czas są używane, choć same się nie zmieniają.

**Uczciwa uwaga o priorytetyzacji:** na dziś upraszczamy ją do zasady **recency-first** - do promptu bierzemy **2 ostatnie epizody** (`recent(2)`), a starsze wygaszamy limitem `max_episodes`. To wystarczy na warsztat. Bardziej wyrafinowaną priorytetyzację (waga epizodu, czas ostatniego użycia, TTL) zostawiamy na pełną ścieżkę.

<details>
<summary>🔧 Dla ciekawych: co jeszcze bywa w tych warstwach (nieobowiązkowe)</summary>

Na produkcji `WorkingMemory` dostaje zwykle jeszcze `scratchpad` (wewnętrzne notatki agenta), `last_error` (np. gdy narzędzie padło), a epizody bywają ważone przez `importance` do priorytetyzacji. Świadomie to dziś pomijamy, żeby nie przeciążać - w pełnej ścieżce (Agentic AI) wchodzimy w to głębiej.
</details>

## Ożywiamy Marka (fakty z Dnia 1) - ten obiekt jest gotowy, tylko go uruchom

In [ ]:
state = AgentState(
    core=CoreMemory(),
    profile=StudentProfile(
        user_id="marek_01",
        name="Marek",
        level="B1",
        goal="Poprowadzić prezentację sprzedażową po angielsku w Nowym Jorku",
        preferences=["woli roleplay", "konkret zamiast wykładów o gramatyce"],
        context_notes="Ma kota Lunę - dobry temat na przełamanie lodów.",
    ),
    progress=LearningProgress(),
    workflow=WorkflowMemory(topic="Otwarcie prezentacji"),
    working=WorkingMemory(),
    episodic=EpisodicMemory(),
)

print("✅ Marek jest! :). Poziom:", state.profile.level, "| Cel:", state.profile.goal)

## Technika KK ("Końskie Klapki") - co to jest i skąd ta nazwa?

Zanim puścimy kod, musimy nazwać jedną technikę bardzo wprost. Nazwałem je krótko **KK** czyli **Końskie Klapki**. Tak, to moja nazwa. Nie znajdziesz tej techniki na przykład w publikacjach, ale w praktyce jest ona skuteczna.

Nazwa jest celowo obrazowa:
- koń w zaprzęgu ma klapki po bokach, żeby **nie rozpraszał się całym otoczeniem**
- widzi tylko ten fragment drogi, który jest ważny **tu i teraz**
- z LLM-em robimy dokładnie to samo

Bo problem jest realny: jeśli model widzi **za dużo naraz**, zaczyna być "nadgorliwy". Chce użyć wszystkiego od razu, wybiega przed proces, zdradza temat za wcześnie albo miesza etapy rozmowy.

<img src="../images/horse.jpg" width="800px" />

**KK** to więc nie (tylko) żart językowy, tylko konkretna technika:

1. **Nie prosisz modelu grzecznie, żeby czegoś nie używał.**
2. **Fizycznie wycinasz to z promptu**, żeby tego po prostu nie widział.
3. Robisz to zwykle przez `if`-y w szablonie (`Jinja2`) sterowane przez `WorkflowMemory`.

W naszym Tutorze będzie to wyglądać tak:
- w fazie `HOMEWORK_CHECK` model **nie widzi jeszcze** tematu roleplay
- dopiero w `PRACTICE` temat staje się widoczny

To właśnie zobaczysz za chwilę w `build_prompt(state)`. `Workflow` to lejce, a **KK to zawężenie pola widzenia modelu**.


```text
   BEZ KLAPEK                          Z KLAPKAMI (KK)
   ─────────────                       ────────────────
   Model widzi WSZYSTKO naraz:         Model widzi tylko to, co teraz ważne:

        🐎  →  😵                            🐎|  →  🎯
     ┌──────────────┐                     ┌──────────────┐
     │ temat        │                     │ FAZA:        │
     │ zadanie      │  wypala             │ HOMEWORK_    │
     │ cały plan    │  za wcześnie        │ CHECK        │
     │ przyszłość   │  ─────────►         │              │
     └──────────────┘  "nadgorliwy"       │ temat = 🚫   │  ukryty
                                          └──────────────┘
                                          (if w szablonie
                                           wycina go fizycznie)
```                                           

## Zanim cokolwiek wyjaśnię - zobacz to w akcji

Mam dla Ciebie gotową maszynę: dwie funkcje (`build_prompt` i `run_turn`), które za chwilę rozbierzemy na części. **Teraz nie czytaj ich literka po literce** - po prostu uruchom obie komórki i patrz na wynik.

Poniżej jest gotowy kod - uruchom, nie musisz teraz czytać. Rozbierzemy za chwilę.

In [ ]:
# SELECT: budujemy prompt z wyselekcjonowanych warstw pamięci
PROMPT_TEMPLATE = Template("""
Jesteś {{ s.core.name }} - {{ s.core.role }}.
Zasady: {{ s.core.prime_directives | join(" | ") }}

UCZEŃ: {{ s.profile.name }} (poziom {{ s.profile.level }})
Cel: {{ s.profile.goal }}
Preferencje: {{ s.profile.preferences | join(", ") }}
{% if s.profile.context_notes %}Notatki: {{ s.profile.context_notes }}{% endif %}

POSTĘP (pamiętaj o tym!):
- Ostatnie błędy do przećwiczenia: {{ s.progress.recent_mistakes | join("; ") or "brak" }}
- Następny krok: {{ s.progress.next_focus or "brak" }}
{% if s.episodic.episodes %}
Z POPRZEDNICH SESJI:
{% for e in s.episodic.recent(2) %}- {{ e.summary }}
{% endfor %}{% endif %}
AKTUALNA FAZA: {{ s.workflow.phase.value }}
{% if s.workflow.phase.value == "HOMEWORK_CHECK" %}👉 Sprawdź pracę domową. NIE zaczynaj nowego tematu.
{% elif s.workflow.phase.value == "PRACTICE" %}👉 Roleplay na temat: "{{ s.workflow.topic }}". Poprawiaj tylko krytyczne błędy.
{% elif s.workflow.phase.value == "WRAP_UP" %}👉 Podsumuj i zadaj krótką pracę domową.
{% endif %}
""")

def build_prompt(state: AgentState) -> str:
    return PROMPT_TEMPLATE.render(s=state).strip()


# WRITE + SELECT: jedna tura rozmowy
class TutorReply(BaseModel):
    reply: str = Field(description="Odpowiedź tutora do ucznia")

def run_turn(state: AgentState, user_message: str) -> str:
    state.working.add("user", user_message)                                   # WRITE
    system_prompt = build_prompt(state)                                       # SELECT
    
    messages = [
        {"role": "system", "content": system_prompt}
    ] + state.working.get_context_window()
    
    result = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        temperature=0,   
        response_model=TutorReply,
    )
    state.working.add("assistant", result.reply)                             # WRITE
    return result.reply

print("✅ Maszyna gotowa. Odpalamy pierwszą turę ⬇️")

## Wchodzimy w fazę `PRACTICE` i robimy jedną turę

In [ ]:
state.workflow.phase = SessionPhase.PRACTICE

reply = run_turn(state, "Hi! Ready when you are.")
print("🤖 TUTOR:", reply)

## Rentgen warstw po jednej turze

Zatrzymajmy się na sekundę. Jeszcze **nie zamykaliśmy sesji**, nie było kompresji, nie było zapisu epizodu. To dobry moment, żeby zobaczyć **które warstwy już ruszyły, a które dopiero ruszą**.

In [ ]:
print("🔬 RENTGEN WARSTW PO 1 TURZE")
print("CORE.role:", state.core.role, "(bez zmian - ale użyte w prompcie)")
print("PROFILE.goal:", state.profile.goal, "(bez zmian - ale użyte w prompcie)")
print("WORKFLOW.phase:", state.workflow.phase.value, "(steruje tym, co model widzi)")
print("WORKING.history entries:", len(state.working.history), "->", [m["role"] for m in state.working.history])
print("EPISODIC.episodes:", len(state.episodic.episodes), "(jeszcze 0 - nie było kompresji)")
print("PROGRESS.next_focus:", state.progress.next_focus or "brak", "(jeszcze brak - nic nie zapisaliśmy)")


## Co się właśnie stało? (to jest ten moment)

Nigdzie nie napisaliśmy w tym poleceniu "użyj imienia Marek i wspomnij o Nowym Jorku". A mimo to Tutor **wiedział, z kim rozmawia i po co**. Skąd? Bo pamięć (obiekt `state`) została **świadomie wstrzyknięta do promptu** przez `build_prompt`.

To jest cała różnica między amnezyjnym chatbotem a agentem, którego projektujesz. Teraz otwórzmy maskę i zobaczmy, jak te dwie funkcje to zrobiły.

Po tym jednym ruchu wydarzyły się trzy ważne rzeczy:

1. **`WorkingMemory` już się zmieniła** - w historii siedzą dwie wiadomości: użytkownika i tutora.
2. **`WorkflowMemory` właśnie pracowała** - faza `PRACTICE` zadecydowała, że model zobaczył temat sesji.
3. **`EpisodicMemory` i `LearningProgress` jeszcze się nie zmieniły** - bo ich moment przychodzi dopiero po kompresji i zapisie epizodu.

## Otwieramy maskę #1: `build_prompt` (operacja Select + KK)

`build_prompt` bierze `state` i zamienia pamięć w tekst promptu. Nie tłumaczę całego szablonu - **patrz tylko na dwie rzeczy**:

> **Stop klatka:**
> 1. **Blok `POSTĘP`** - to tutaj z pamięci wpadają do promptu ostatnie błędy i następny krok. To on niesie "pamięta postęp".
> 2. **Blok `FAZA` z `{% if %}`** - to jest **KK ("Końskie Klapki")**. Zauważ: temat sesji (`topic`) pokazujemy **tylko** w fazie `PRACTICE`. W `HOMEWORK_CHECK` model **fizycznie go nie widzi** - więc nie ma jak przedwcześnie nim wypalić.

To jest sedno świadomego zasilania: nie wrzucamy wszystkiego do promptu. **Wybieramy**, co model widzi i kiedy.

## Otwieramy maskę #2: `run_turn` (operacja Write + Select)

`run_turn` to serce agenta - jedna tura request-response. Znów: **patrz tylko na 3 linie**:

> **Stop klatka:**
> 1. `state.working.add("user", user_message)` → **Write**: zapisujemy pytanie ucznia do pamięci.
> 2. `build_prompt(state)` → **Select**: budujemy prompt z pamięci (to właśnie widziałeś wyżej).
> 3. `state.working.add("assistant", result.reply)` → **Write**: zapisujemy odpowiedź z powrotem.

Cała reszta (wywołanie modelu, sklejenie wiadomości) to hydraulika. Te 3 linie to mechanika pamięci.

## Otwieramy maskę #3: `compress_episode` (operacja Compress)

Pamiętasz szczebel 3 z Dnia 1? *"Niech model streszcza rozmowę"* - i niespodzianka: **naiwne streszczanie zawodzi** (dryfuje, gubi sedno, wprowadza sprzeczności). Obiecałem pokazać, jak robić to dobrze.

Sekret w jednej linii:

> **Stop klatka:** `response_model=Episode`
> Nie prosimy modelu o luźne "streść to" (dryf). Każemy mu **wypełnić formularz** (`Episode`): podsumowanie, fakty, błędy, następny krok. To jest **inteligentna niszczarka** - z tysiąca słów zostaje czysta esencja, którą da się zapisać do bazy.

Uruchom i zobacz efekt:

In [ ]:
def compress_episode(state: AgentState) -> Episode:
    transcript = "\n".join(f"{m['role']}: {m['content']}" for m in state.working.history)
    messages = [
        {"role": "system", "content": "Jesteś inżynierem pamięci. Wyciągnij esencję sesji do struktury. Ignoruj small talk, chyba że zawiera trwałe fakty o uczniu."},
        {"role": "user", "content": transcript},
    ]
    return client.chat.completions.create(
        model="gpt-4o-mini",
        response_model=Episode,
        messages=messages,
        temperature=0,
    )

episode = compress_episode(state)
print(episode.model_dump_json(indent=2))

## 🧠 Zadanie 2.1: Poczuj KK w palcach

KK było już widać w `build_prompt`. Teraz czas zrobić to samodzielnie na mini-przykładzie - to najszybszy sposób, żeby ta mechanika weszła w palce.

Napisz krótki szablon Jinja2, który **ukrywa temat sesji poza fazą PRACTICE**:
- `HOMEWORK_CHECK` → instrukcja: sprawdź pracę domową. **Temat NIE jest widoczny.**
- `PRACTICE` → dopiero teraz pokaż temat.

Potem wyrenderuj go dla obu faz, żeby zobaczyć różnicę.

> 🛟 **Dobra wiadomość:** poniżej jest bezpieczny szkielet. Komórka uruchomi się nawet z placeholderami `TODO`, więc nic się nie wywali. Ale pełny efekt KK zobaczysz dopiero po uzupełnieniu obu miejsc.

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;"> 👉 Kliknij tutaj, aby zobaczyć podpowiedź 👈 </summary>
<p>

Dostęp do wartości fazy: `{{ state.workflow.phase.value }}`. Zmiana fazy w teście: `state.workflow.phase = SessionPhase.PRACTICE`.

```python
kk_template = Template("""
Faza: {{ state.workflow.phase.value }}
{% if state.workflow.phase.value == "HOMEWORK_CHECK" %}
    👉 Sprawdź pracę domową. NIE zdradzaj tematu.
{% elif state.workflow.phase.value == "PRACTICE" %}
    👉 Temat: {{ state.workflow.topic }}
{% endif %}
""")
```

</p>

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;"> 👉 Kliknij tutaj, aby zobaczyć rozwiązanie 👈 </summary>
<p>

```python
kk_template = Template("""
Uczeń: {{ state.profile.name }} | Faza: {{ state.workflow.phase.value }}
{% if state.workflow.phase.value == "HOMEWORK_CHECK" %}
    👉 Sprawdź pracę domową. NIE zaczynaj nowego tematu, nie zdradzaj, co dziś ćwiczycie.
{% elif state.workflow.phase.value == "PRACTICE" %}
    👉 Prowadź roleplay na temat: "{{ state.workflow.topic }}".
{% endif %}
""")

state.workflow.phase = SessionPhase.HOMEWORK_CHECK
print("=== FAZA 1: HOMEWORK_CHECK (temat ukryty) ===")
print(kk_template.render(state=state))

state.workflow.phase = SessionPhase.PRACTICE
print("=== FAZA 2: PRACTICE (temat widoczny) ===")
print(kk_template.render(state=state))
```

<br />
W pierwszym wydruku modelu <b>fizycznie brakuje</b> tematu. Nie zhalucynuje go, choćby chciał. To Ty rządzisz kontekstem. Poczuj moc :).

</p>
</details>
</details>

In [ ]:
## YOUR CODE HERE

## 🧠 Zadanie 2.2: Zamknij pętlę (uzupełnij 3 linie)

Mamy `Write` (`run_turn`), `Select` (`build_prompt`) i `Compress` (`compress_episode`). Brakuje ostatniego ogniwa: **epizod musi wrócić do stanu**. Bez tego cała kompresja ląduje w koszu, a Tutor jutro znów zaczyna od zera.

Poniższa funkcja jest prawie gotowa. **Uzupełnij tylko 3 oznaczone linie** (`# ✏️`). Reszta jest zrobiona za Ciebie.

> ⚠️ Klasyczna pułapka z produkcji: model kompresji zadziałał **idealnie**, zwrócił błędy... a ktoś zapomniał zapisać je do stanu. Efekt: agent pyta o to samo w kółko. To ten krok robi z chatbota agenta, który pamięta postęp.

> 🛑 **WAŻNE:** najpierw **uzupełnij 3 linie i uruchom tę komórkę**, dopiero potem odpalaj pełną pętlę. Zostawiłem bezpieczne wartości domyślne, więc komórka nie wywali błędu nawet nieuzupełniona - ale wtedy postęp zostanie pusty i nie zobaczysz pełnego efektu.

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;"> 👉 Kliknij tutaj, aby zobaczyć rozwiązanie 👈 </summary>
<p>

```python
def apply_episode_to_state(state: AgentState, episode: Episode) -> AgentState:
    state.episodic.add(episode)
    mistakes_as_text = [f"{m.user_phrase} -> {m.correction} ({m.rule})" for m in episode.mistakes]

    state.progress.recent_mistakes = mistakes_as_text[:5]   # ✏️ 1
    state.progress.next_focus = episode.next_step           # ✏️ 2
    state.progress.homework_status = "assigned"             # ✏️ 3
    return state
```

</p>
</details>

In [ ]:
## YOUR CODE HERE

def apply_episode_to_state(state: AgentState, episode: Episode) -> AgentState:
    # (gotowe) zapisz epizod do warstwy episodic - wygaszanie dzieje się w add()
    state.episodic.add(episode)

    # (gotowe) zamień błędy na czytelne stringi
    mistakes_as_text = [f"{m.user_phrase} -> {m.correction} ({m.rule})" for m in episode.mistakes]

    # ✏️ 1: podmień []  ->  mistakes_as_text[:5]
    state.progress.recent_mistakes = []

    # ✏️ 2: podmień ""  ->  episode.next_step
    state.progress.next_focus = ""

    # ✏️ 3: podmień "none"  ->  "assigned"
    state.progress.homework_status = "none"

    return state

## Pełna pętla: `state before → reply → compression → state after → nowy prompt`

To jest cały dzień 2 w jednym przebiegu - dowód, że masz **działający, samoaktualizujący się model stanu**. Tym razem patrz nie tylko na `recent_mistakes`, ale też na:

- `working.history` → czy bieżąca rozmowa rośnie?
- `episodic.episodes` → czy sesja została zapisana?
- `progress.next_focus` → czy postęp wrócił do stanu?
- `build_prompt(state)` → czy następny prompt jest już bogatszy?

A na końcu pokażę Ci następny prompt - z pamięcią, która właśnie wróciła.


```

              ┌──────────────────────────────────────────────┐
              │            PĘTLA PAMIĘCI (Dzień 2)           │
              └──────────────────────────────────────────────┘

   Uczeń pisze  ─────────────►  🔴 WRITE
                                 zapisz wiadomość do Working
                                        │
                                        ▼
                                🔵 SELECT  (build_prompt)
                                 zbierz pamięć → prompt
                                        │
                                        ▼
                                 🤖 Model odpowiada
                                        │
                                        ▼
                                🔴 WRITE
                                 zapisz odpowiedź do Working
                                        │
                        (... rozmowa trwa, tura po turze ...)
                                        │
                             ─── KONIEC SESJI ───
                                        │
                                        ▼
                                🟢 COMPRESS  (compress_episode)
                                 rozmowa → czysty Episode
                                        │
                                        ▼
                                🔴 WRITE  (apply_episode_to_state)
                                 epizod + postęp wracają do stanu
                                        │
                                        ▼
                           NASTĘPNY PROMPT JEST BOGATSZY
                                        │
                                        └──► (i pętla zaczyna się od nowa)

```

In [ ]:
# dla czytelności zaczynamy tę mini-sesję od czystej pamięci roboczej
state.working = WorkingMemory()

# 1. STATE BEFORE (postęp jeszcze pusty)
print("STATE BEFORE  → recent_mistakes:", state.progress.recent_mistakes, "| epizody:", len(state.episodic.episodes))
print("              → working entries:", len(state.working.history), "| faza:", state.workflow.phase.value)

# 2. MODEL REPLY (jedna tura na żywo, z błędem do wyłapania)
state.workflow.phase = SessionPhase.PRACTICE
reply = run_turn(state, "I want say: welcome everybody on the meeting.")
print("\nTUTOR:", reply)

# 3. COMPRESSION (koniec sesji)
episode = compress_episode(state)

# 4. APPLY → STATE AFTER (postęp się zaktualizował)
apply_episode_to_state(state, episode)
print("\nSTATE AFTER   → recent_mistakes:", state.progress.recent_mistakes)
print("              → next_focus:", state.progress.next_focus, "| epizody:", len(state.episodic.episodes))
print("              → working entries:", len(state.working.history), "| faza:", state.workflow.phase.value)

# 5. NASTĘPNY PROMPT (dowód: pamięć wróciła i zasila kontekst)
print("\n" + "=" * 55)
print("NASTĘPNY PROMPT - zobacz sekcję POSTĘP:")
print("=" * 55)
print(build_prompt(state))

## Co się właśnie stało? (sedno całego dnia)

Przed chwilą `recent_mistakes` było puste, a epizodów było `0`. Po **jednym** pełnym obiegu pętli stan **sam się zaktualizował**: błędy Marka wylądowały w `progress`, epizod trafił do `episodic`, a `next_focus` wie, co ćwiczyć dalej.

I najważniejsze - zobaczyłeś to na własne oczy w ostatnim wydruku: **następny prompt jest już bogatszy**. Sekcja `POSTĘP` sama wypełniła się błędami z zamkniętej sesji. To jest zamknięta pętla pamięci:

> **Write → Select → (rozmowa) → Compress → Write → i znów Select z bogatszą pamięcią.**

Dokładnie to odróżnia agenta, który "pamięta postęp", od chatbota, który wita Cię jak obcego. Mamy to w czystym Pythonie, bez żadnej bazy danych.

<div class="alert alert-block alert-danger">

<h2>✅ Zaliczenie Dnia 2</h2>
<h4>Co masz teraz zrobić? (3 kroki)</h4>

<ul>
    <ol>📸 <b>Zrób screenshot pełnej pętli:</b> wynik ostatniej komórki - ma być widać <code>STATE BEFORE</code>, <code>TUTOR</code>, <code>STATE AFTER</code> z wypełnionym <code>recent_mistakes</code> oraz <code>NASTĘPNY PROMPT</code> z sekcją POSTĘP.</ol>
    <ol>💬 <b>Wrzuć na Slacka:</b> na kanał <code><a href="https://dataworkshopclub.slack.com/archives/C0ADAKBRX3N">#dwthon_memory_day2_done</a></code>.</ol>
    <ol>🆘 <b>Brak dostępu do Slacka?</b> Napisz: <code>hello@dataworkshop.eu</code></ol>
</ul>
</div>

### 🎁 Punkty

- Za dzisiaj zbierasz kolejne **500 Punktów Bonusowych**.
- Za oddane zadanie bot wysyła Ci na mailaю
- Komplet z 3 dni = **certyfikat** + coś, o czym powiem jutro. 😉

## Podsumowanie Dnia 2

Dziś przestaliśmy gadać o pamięci - zbudowaliśmy działającą pętlę:

1. **4 operacje** - Write, Select, Compress, Isolate - każdą zobaczyłeś w kodzie.
2. **Pamięć to konkretny kod** (Pydantic). Czego nie ma w strukturze, tego agent nie pamięta.
3. **`AgentState` jako jedno źródło prawdy** - z izolacją warstw.
4. **`LearningProgress`** - dzięki niemu Tutor pamięta nie tylko fakty, ale i **postęp**.
5. **`build_prompt` = świadome zasilanie** - selekcja warstw + KK (fizyczne odcięcie tego, czego model nie powinien teraz widzieć).
6. **Kompresja strukturalna** - "wypełnij formularz" bije naiwne streszczanie na głowę.
7. **Zamknięta pętla** - epizod wraca do stanu, aktualizuje postęp i zasila następny prompt.

## Co dalej? (Dzień 3)

Masz działający model stanu - ale żyje w komórkach notebooka. Jutro **ożywiamy to w Gradio**:
- porozmawiasz z Tutorem na żywo,
- zobaczysz jego pamięć w prawym panelu (rentgen stanu, aktualizujący się na oczach),
- przełączysz `user_id` i zobaczysz, jak pamięć się zmienia,
- i - to jest ten moment - **zamkniesz sesję, otworzysz nową, a Tutor Cię rozpozna**: przywoła kontekst i postęp, zamiast zaczynać od zera.

A jeśli poczujesz niedosyt - taki był plan. Na spokojnie. To fundament. Pełne 7 warstw, pamięć proceduralna (*skills*), sub-agenci i konsolidacja w tle to głęboka woda, w którą wchodzimy w **Agentic AI**. Najpierw jednak dowieźmy jutro artefakt. Do zobaczenia. 👋

## Przydatne linki

1. [Pydantic - dokumentacja](https://docs.pydantic.dev/)
2. [Jinja2 - dokumentacja](https://jinja.palletsprojects.com/)
3. [Anthropic - Effective context engineering for AI agents](https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents)
4. [OpenAI - Structured Outputs](https://platform.openai.com/docs/guides/structured-outputs)